# XX Simulation and Reinforcement Learning

In [31]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Import packages                         #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import pandas as pd
import simpy

In [32]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Reset working directory                 #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import __main__
_nb = getattr(__main__, "__vsc_ipynb_file__", None) or os.environ.get("JPY_SESSION_NAME")
_start = Path(_nb).resolve().parent if _nb else Path.cwd()
os.chdir(next(p for p in [_start, *_start.parents] if (p / "pyproject.toml").exists()))
print(f"Working directory: {os.getcwd()}")

Working directory: /Users/hendrik/Coding/Master/AAA/AAA_TA_2026


## Components of the simulation

**Entities:**
- Electric taxi with state batterie charging
- Charging Agent

**Events:**
- Arrival 
- begin of every 15min interval
- Departure

**Activities:**
- Charging Interval
- Working day beginning after depature

**Resources:** 
- Battery
- Charger

In [33]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Constants for Simulation Configuration  #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

RANDOM_SEED = 42          # Seed for random number generator for reproducibility
N_STEPS = 8
T_ARRIVAL = 14
T_DEPART = 16

BATTERY_CAPACITY = 60
SOC_INIT = 0  # SOC = state of charge; Initial SOC of the battery at the start

P_MAX = 22 # Maximum charging power 
CHARGING_LEVELS = {'zero': 0, 'low': 7, 'medium': 14, 'high': 22}  # Available charging levels

ALPHA = 0.9

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Process functions                       #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import numpy as np

# ============================================================================
# 1. STATE REPRESENTATION
# ============================================================================

def get_state(soc, time_step):
    return (round(soc, 2), time_step)


# ============================================================================
# 2. ACTION SPACE
# ============================================================================

def power_from_action(action_name):
    return CHARGING_LEVELS.get(action_name, 0)


# ============================================================================
# 3. ENVIRONMENT DYNAMICS
# ============================================================================

def charge_battery(soc, power, duration_hours=0.25):
    energy_added = power * duration_hours
    new_soc = min(soc + energy_added, BATTERY_CAPACITY)
    return new_soc


def generate_energy_demand(mu=30, sigma=5):
    demand = np.random.normal(mu, sigma)
    return max(0, demand)  # Ensure non-negative


# ============================================================================
# 4. COST FUNCTION
# ============================================================================

def charging_cost(power, alpha=ALPHA):
    if power == 0:
        return 0
    return alpha * np.exp(power)


# ============================================================================
# 5. REWARD FUNCTION
# ============================================================================

def calculate_reward(action, soc_before, soc_after, time_step, energy_demand=None):
    power = power_from_action(action)
    cost = charging_cost(power)
    reward = -cost  # Penalize high costs

    # At departure (time_step == N_STEPS - 1), check if enough energy
    if time_step == N_STEPS - 1 and energy_demand is not None:
        if soc_after < energy_demand:
            reward -= 1000  # Large penalty for insufficient charge

    return reward


# ============================================================================
# 6. SIMULATION STEP
# ============================================================================

def step(soc, action, time_step, energy_demand=None):
    power = power_from_action(action)
    soc_new = charge_battery(soc, power)

    done = (time_step == N_STEPS - 1)
    reward = calculate_reward(action, soc, soc_new, time_step, energy_demand)

    return soc_new, reward, done


# ============================================================================
# 7. HELPER: CHECK IF FEASIBLE
# ============================================================================

def is_feasible(soc_at_departure, energy_demand):
    return soc_at_departure >= energy_demand

## Environment Loop

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# SimPy Environment (Discrete Event Sim)  #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

class Taxi:
    """Electric taxi with battery and charging state."""
    
    def __init__(self, env, policy_fn=None):
        self.env = env
        self.policy_fn = policy_fn  # Function to decide charging action
        self.soc = SOC_INIT
        self.energy_demand = None
        self.feasible = False
        self.episode_history = []
        self.total_cost = 0
        
    def run(self):
        """Main process: arrival → charging loop → departure."""
        
        # Event: Arrival at 2pm
        yield self.env.timeout(T_ARRIVAL)
        print(f"[{self.env.now}:00] Taxi arrives at home. Battery SOC: {self.soc:.2f} kWh")
        
        # Charging phase: make decisions every 15 minutes
        for t in range(N_STEPS):
            # Wait 15 minutes (next decision point)
            yield self.env.timeout(0.25)  # 15 min = 0.25 hour
            
            state = get_state(self.soc, t)
            
            # Agent decides action
            if self.policy_fn is None:
                action = np.random.choice(list(CHARGING_LEVELS.keys()))
            else:
                action = self.policy_fn(state)
            
            # Execute charging step
            soc_new, reward, done = step(self.soc, action, t)
            power = power_from_action(action)
            
            # Record history
            self.episode_history.append({
                'time_step': t,
                'soc_before': self.soc,
                'action': action,
                'power_kw': power,
                'soc_after': soc_new,
                'cost': charging_cost(power),
                'reward': reward
            })
            
            self.total_cost += charging_cost(power)
            self.soc = soc_new
            
            print(f"[{self.env.now:.2f}:00] Step {t}: Action={action:7s}, Power={power:2.0f}kW, SOC={self.soc:.2f}kWh")
        
        # Event: Departure at 4pm
        yield self.env.timeout(0)
        print(f"[{self.env.now}:00] Ready to depart. Final SOC: {self.soc:.2f} kWh")
        
        # Generate energy demand (stochastic)
        self.energy_demand = generate_energy_demand(mu=30, sigma=5)
        self.feasible = is_feasible(self.soc, self.energy_demand)
        
        print(f"\n{'='*70}")
        print(f"Energy demand: {self.energy_demand:.2f} kWh")
        print(f"Feasible (enough charge)? {self.feasible}")
        print(f"Total cost: {self.total_cost:.2f}")
        print(f"{'='*70}\n")


def run_simulation(policy_fn=None, seed=RANDOM_SEED):
    np.random.seed(seed)
    
    env = simpy.Environment()
    taxi = Taxi(env, policy_fn=policy_fn)
    env.process(taxi.run())
    env.run()
    
    return taxi


# ============================================================================
# Run simulation
# ============================================================================

print("\n" + "=" * 70)
print("SIMPY DISCRETE EVENT SIMULATION - Random Policy")
print("=" * 70 + "\n")

taxi = run_simulation(policy_fn=None)

# Print charging history
print("\nCharging History:")
print("-" * 70)
history_df = pd.DataFrame(taxi.episode_history)
print(history_df.to_string(index=False))
print("-" * 70)


SIMPY DISCRETE EVENT SIMULATION - Random Policy

[14:00] Taxi arrives at home. Battery SOC: 0.00 kWh
[14.25:00] Step 0: Action=medium , Power=14kW, SOC=3.50kWh
[14.50:00] Step 1: Action=high   , Power=22kW, SOC=9.00kWh
[14.75:00] Step 2: Action=zero   , Power= 0kW, SOC=9.00kWh
[15.00:00] Step 3: Action=medium , Power=14kW, SOC=12.50kWh
[15.25:00] Step 4: Action=medium , Power=14kW, SOC=16.00kWh
[15.50:00] Step 5: Action=high   , Power=22kW, SOC=21.50kWh
[15.75:00] Step 6: Action=zero   , Power= 0kW, SOC=21.50kWh
[16.00:00] Step 7: Action=zero   , Power= 0kW, SOC=21.50kWh
[16.0:00] Ready to depart. Final SOC: 21.50 kWh

Energy demand: 28.83 kWh
Feasible (enough charge)? False
Total cost: 6456090154.60


Charging History:
----------------------------------------------------------------------
 time_step  soc_before action  power_kw  soc_after         cost        reward
         0         0.0 medium        14        3.5 1.082344e+06 -1.082344e+06
         1         3.5   high        22   

In [39]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Monte Carlo Simulation (Random Policy)  #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

def run_many_episodes(n_episodes=100, policy_fn=None, verbose=False):
    results = []
    
    for episode_id in range(n_episodes):
        taxi = run_simulation(policy_fn=policy_fn, seed=RANDOM_SEED + episode_id)
        
        results.append({
            'episode': episode_id,
            'soc_final': taxi.soc,
            'energy_demand': taxi.energy_demand,
            'feasible': taxi.feasible,
            'total_cost': taxi.total_cost
        })
        
        if verbose and (episode_id + 1) % max(1, n_episodes // 10) == 0:
            print(f"  Completed {episode_id + 1}/{n_episodes} episodes...")
    
    return pd.DataFrame(results)


# ============================================================================
# Run 100 episodes with random policy
# ============================================================================

print("\n" + "=" * 70)
print("MONTE CARLO ANALYSIS - 100 Episodes (Random Policy)")
print("=" * 70 + "\n")

print("Running 100 simulations...")
results_df = run_many_episodes(n_episodes=100, policy_fn=None, verbose=False)

# Compute statistics
n_feasible = results_df['feasible'].sum()
success_rate = (n_feasible / len(results_df)) * 100
avg_cost = results_df['total_cost'].mean()
avg_soc_final = results_df['soc_final'].mean()
avg_demand = results_df['energy_demand'].mean()
std_demand = results_df['energy_demand'].std()

print("\n" + "=" * 70)
print("RESULTS SUMMARY")
print("=" * 70)
print(f"Success Rate (feasible):        {success_rate:.1f}% ({n_feasible}/{len(results_df)})")
print(f"Average Final SOC:              {avg_soc_final:.2f} kWh")
print(f"Average Energy Demand:          {avg_demand:.2f} ± {std_demand:.2f} kWh")
print(f"Average Total Cost:             {avg_cost:.2f}")
print(f"Min Cost:                       {results_df['total_cost'].min():.2f}")
print(f"Max Cost:                       {results_df['total_cost'].max():.2f}")
print("=" * 70)

# Show first few episodes
print("\nFirst 10 episodes:")
print(results_df.head(10).to_string(index=False))

# Show failure cases
failures = results_df[~results_df['feasible']]
if len(failures) > 0:
    print(f"\nFailed episodes ({len(failures)} total):")
    print(failures.to_string(index=False))


MONTE CARLO ANALYSIS - 100 Episodes (Random Policy)

Running 100 simulations...
[14:00] Taxi arrives at home. Battery SOC: 0.00 kWh
[14.25:00] Step 0: Action=medium , Power=14kW, SOC=3.50kWh
[14.50:00] Step 1: Action=high   , Power=22kW, SOC=9.00kWh
[14.75:00] Step 2: Action=zero   , Power= 0kW, SOC=9.00kWh
[15.00:00] Step 3: Action=medium , Power=14kW, SOC=12.50kWh
[15.25:00] Step 4: Action=medium , Power=14kW, SOC=16.00kWh
[15.50:00] Step 5: Action=high   , Power=22kW, SOC=21.50kWh
[15.75:00] Step 6: Action=zero   , Power= 0kW, SOC=21.50kWh
[16.00:00] Step 7: Action=zero   , Power= 0kW, SOC=21.50kWh
[16.0:00] Ready to depart. Final SOC: 21.50 kWh

Energy demand: 28.83 kWh
Feasible (enough charge)? False
Total cost: 6456090154.60

[14:00] Taxi arrives at home. Battery SOC: 0.00 kWh
[14.25:00] Step 0: Action=zero   , Power= 0kW, SOC=0.00kWh
[14.50:00] Step 1: Action=zero   , Power= 0kW, SOC=0.00kWh
[14.75:00] Step 2: Action=high   , Power=22kW, SOC=5.50kWh
[15.00:00] Step 3: Action=lo